# 南水北调中线数据探索

In [ ]:
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei"]
plt.rcParams["axes.unicode_minus"] = False

from seehydro.acquisition.route import RouteDataLoader, load_route
from seehydro.utils.config import load_config
cfg = load_config()
print("OK")

## 1. 加载线路数据

In [ ]:
loader = RouteDataLoader()
try:
    route = loader.from_geojson("data/route/snbd_centerline.geojson")
except FileNotFoundError:
    route = loader.from_osm()
    loader.save(route, "data/route/snbd_centerline.geojson")
print(loader.get_route_info(route))
route.head()

## 2. 线路可视化

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))
route.plot(ax=ax, color="blue", linewidth=2)
ax.set_title("南水北调中线")
plt.show()

## 3. 缓冲区

In [ ]:
buf = loader.buffer(route, width_m=2000)
segments = loader.split_segments(route, length_m=10000)
print(f"分段数: {len(segments)}")
fig, ax = plt.subplots(figsize=(10, 12))
buf.plot(ax=ax, alpha=0.3, color="lightblue")
route.plot(ax=ax, color="blue", linewidth=2)
plt.show()

## 4. 影像浏览

In [ ]:
s2_dir = Path("data/sentinel2")
if s2_dir.exists():
    tifs = list(s2_dir.rglob("*.tif"))
    print(f"找到 {len(tifs)} 个影像")
else:
    print("运行 seehydro download sentinel2 下载影像")